[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/main/ETL_inicial/ETL_inicial.ipynb)

# ETL Paderborn (64 kHz) — Formato Compatible con C1 y C3

Descarga archivos `.rar` de fallas reales del dataset de Paderborn desde Zenodo, extrae ventanas de `2048` muestras a `64 kHz` y aplica **exactamente el mismo pipeline de features y tratamiento** que `paderborn_features_tratadas.csv` y los notebooks de los circuitos `C1` y `C3`:

1. **Extracción de 71 features + `Label`** (mismos nombres y orden que `paderborn_features_tratadas.csv`):
   - Operativas (3): `RPM`, `Torque_Nm`, `Fuerza_N`
   - Estadísticas (4): `RMS`, `Kurtosis` (exceso de Fisher), `Skewness`, `Crest_Factor`
   - Espectrales (64): `fft_0` ... `fft_63` (`|rfft(x - mean)|[:64]` normalizado L2)
2. **Tratamiento de datos:** limpieza `NaN/Inf` → `log1p(|fft|)` → **Clipping `[P1, P99]`** (usando los límites sanos de `paderborn_features_tratadas.csv`) → **`RobustScaler`** (`fit` exclusivo en `X_train` sano con `random_state=42`) → **Zero-Padding (71 → 128)** → **Normalización L2** por fila (7 qubits, Amplitude Encoding).

> **Hallazgo importante sobre la señal de entrada:** Al inspeccionar `paderborn_features_tratadas.csv`, sus estadísticas (`RMS ≈ 0.90` a `0.1 Nm` y `1.76` a `0.7 Nm`, `Kurtosis ≈ -1.49`, `Crest_Factor ≈ 1.57`, y picos FFT a `60 Hz` / `100 Hz` en `fft_2` / `fft_3`) corresponden al canal **`phase_current_1`** (64 kHz), no a `vibration_1`. Por defecto `CANAL_SENAL = 'phase_current_1'` para mantener compatibilidad estadística exacta con el `RobustScaler` y los modelos ya entrenados; puedes cambiarlo a `'vibration_1'` en la celda de configuración si decides reentrenar desde cero con acelerometría.

In [ ]:
# ==============================================================================
# CELDA 1: ENTORNO, RUTAS Y CONFIGURACIÓN
# ==============================================================================
import os
import sys
import shutil
import urllib.request
import subprocess
import numpy as np
import pandas as pd
import scipy.io
from scipy.stats import kurtosis, skew
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

try:
    import rarfile
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rarfile"])
    import rarfile

if sys.platform.startswith("linux") and not shutil.which("unrar"):
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)

# Detección automática Colab vs local y montaje de Google Drive
EN_COLAB = os.path.exists("/content") and ("google.colab" in sys.modules)
GUARDAR_EN_DRIVE = True
CARPETA_DRIVE = "Paderborn_QML"

WORK_DIR = "/content" if EN_COLAB else os.getcwd()
TEMP_DIR = os.path.join(WORK_DIR, "temp_paderborn_downloads")
os.makedirs(TEMP_DIR, exist_ok=True)

DRIVE_DIR = None
if EN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive") and not os.path.exists("/content/drive/My Drive"):
        drive.mount("/content/drive")
    drive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    DRIVE_DIR = os.path.join(drive_root, CARPETA_DRIVE)
    os.makedirs(DRIVE_DIR, exist_ok=True)

# Buscar el dataset sano de referencia (necesario para P1/P99 y RobustScaler)
RUTAS_REF = [
    os.path.join(WORK_DIR, "paderborn_features_tratadas.csv"),
    os.path.join(WORK_DIR, "..", "paderborn_features_tratadas.csv"),
    os.path.join(WORK_DIR, "..", "..", "paderborn_features_tratadas.csv"),
    "/content/paderborn_features_tratadas.csv",
    "/content/drive/MyDrive/Paderborn_QML/paderborn_features_tratadas.csv",
    "/content/drive/My Drive/Paderborn_QML/paderborn_features_tratadas.csv",
    "/content/drive/MyDrive/Paderborn_QML/VERSION CON ATIPICOS TRATADOS/paderborn_features_tratadas.csv",
]
RUTA_REF = next((os.path.abspath(r) for r in RUTAS_REF if os.path.exists(r)), None)
if RUTA_REF is None:
    raise FileNotFoundError(
        "No se encontró 'paderborn_features_tratadas.csv'. "
        "Se requiere para extraer los límites de clipping [P1, P99] y ajustar el RobustScaler con X_train sano."
    )

# Parámetros del ETL y del pipeline
WINDOW_SIZE = 2048
N_FFT_BINS = 64
DIM_CUANTICA = 128            # 2^7 amplitudes para 7 qubits
RANDOM_STATE = 42             # Idéntico al split 70/15/15 de C1 y C3
CANAL_SENAL = "phase_current_1"  # 'phase_current_1' (coincide con paderborn_features_tratadas.csv) | 'vibration_1'
APLICAR_CLIPPING = True       # Acota signal_cols a [P1, P99] del conjunto sano de referencia

# Archivos de salida
OUT_TRATADAS = os.path.join(WORK_DIR, "paderborn_fallas_tratadas.csv")
OUT_ROBUST = os.path.join(WORK_DIR, "paderborn_fallas_con_robust_scaler.csv")
OUT_CUANTICO = os.path.join(WORK_DIR, "paderborn_fallas_cuantico_128.npz")
OUT_METADATOS = os.path.join(WORK_DIR, "paderborn_fallas_metadatos.csv")

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
if DRIVE_DIR:
    print(f"Carpeta en Google Drive: {DRIVE_DIR}")
print(f"Referencia sana: {RUTA_REF}")
print(f"Canal seleccionado: {CANAL_SENAL} | Ventana: {WINDOW_SIZE} | Clipping: {APLICAR_CLIPPING}")

In [ ]:
# ==============================================================================
# CELDA 2: CATÁLOGO DE ZENODO Y SELECCIÓN DE FALLAS REALES
# ==============================================================================
ZENODO_CATALOG = {
    # --- MOTORES SANOS (NORMALES) ---
    "K001.rar": "https://zenodo.org/records/15845309/files/K001.rar?download=1",
    "K002.rar": "https://zenodo.org/records/15845309/files/K002.rar?download=1",
    "K003.rar": "https://zenodo.org/records/15845309/files/K003.rar?download=1",
    "K004.rar": "https://zenodo.org/records/15845309/files/K004.rar?download=1",
    "K005.rar": "https://zenodo.org/records/15845309/files/K005.rar?download=1",
    "K006.rar": "https://zenodo.org/records/15845309/files/K006.rar?download=1",

    # --- FALLAS ARTIFICIALES (EDM) ---
    "KA01.rar": "https://zenodo.org/records/15845309/files/KA01.rar?download=1",
    "KA03.rar": "https://zenodo.org/records/15845309/files/KA03.rar?download=1",
    "KA05.rar": "https://zenodo.org/records/15845309/files/KA05.rar?download=1",
    "KA07.rar": "https://zenodo.org/records/15845309/files/KA07.rar?download=1",
    "KA08.rar": "https://zenodo.org/records/15845309/files/KA08.rar?download=1",
    "KA09.rar": "https://zenodo.org/records/15845309/files/KA09.rar?download=1",
    "KI01.rar": "https://zenodo.org/records/15845309/files/KI01.rar?download=1",
    "KI03.rar": "https://zenodo.org/records/15845309/files/KI03.rar?download=1",
    "KI05.rar": "https://zenodo.org/records/15845309/files/KI05.rar?download=1",
    "KI07.rar": "https://zenodo.org/records/15845309/files/KI07.rar?download=1",
    "KI08.rar": "https://zenodo.org/records/15845309/files/KI08.rar?download=1",
    "KI09.rar": "https://zenodo.org/records/15845309/files/KI09.rar?download=1",

    # --- FALLAS REALES (FATIGA / VIDA ACELERADA) ---
    "KA04.rar": "https://zenodo.org/records/15845309/files/KA04.rar?download=1",
    "KA15.rar": "https://zenodo.org/records/15845309/files/KA15.rar?download=1",
    "KA16.rar": "https://zenodo.org/records/15845309/files/KA16.rar?download=1",
    "KA22.rar": "https://zenodo.org/records/15845309/files/KA22.rar?download=1",
    "KA30.rar": "https://zenodo.org/records/15845309/files/KA30.rar?download=1",
    "KI04.rar": "https://zenodo.org/records/15845309/files/KI04.rar?download=1",
    "KI14.rar": "https://zenodo.org/records/15845309/files/KI14.rar?download=1",
    "KI16.rar": "https://zenodo.org/records/15845309/files/KI16.rar?download=1",
    "KI18.rar": "https://zenodo.org/records/15845309/files/KI18.rar?download=1",
    "KI21.rar": "https://zenodo.org/records/15845309/files/KI21.rar?download=1",
    "KB23.rar": "https://zenodo.org/records/15845309/files/KB23.rar?download=1",
    "KB24.rar": "https://zenodo.org/records/15845309/files/KB24.rar?download=1",
    "KB27.rar": "https://zenodo.org/records/15845309/files/KB27.rar?download=1",
}

# Selección por defecto: 1 rodamiento real por cada tipo de falla (Pista Exterior, Interior, Combinada).
# Nota: NO incluyas K001..K006 para evaluar sanos; esas ventanas ya están en paderborn_features_tratadas.csv.
# Para evaluar sanos no vistos, usa X_test_q del split 70/15/15.
ARCHIVOS_A_DESCARGAR = [
    "KA04.rar",  # Falla real en pista exterior (fatiga)
    "KI04.rar",  # Falla real en pista interior (fatiga)
    "KB23.rar",  # Falla real combinada (fatiga)
]

In [ ]:
# ==============================================================================
# CELDA 3: DESCARGA Y DESCOMPRESIÓN MULTIPLATAFORMA (COLAB / WINDOWS / LINUX)
# ==============================================================================
def extraer_rar(rar_path, target_folder):
    """Descomprime un .rar usando unrar (Linux), tar (Windows) o rarfile."""
    if shutil.which("unrar"):
        res = subprocess.run(["unrar", "x", "-o+", rar_path, target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "unrar"

    if shutil.which("tar"):
        res = subprocess.run(["tar", "-xf", rar_path, "-C", target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "tar"

    with rarfile.RarFile(rar_path) as rf:
        rf.extractall(path=target_folder)
    return "rarfile"


def download_and_extract_selected(file_list, catalog, target_folder):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print(f"\n[1/4] Descargando {len(file_list)} archivos en {target_folder}...")

    for filename in file_list:
        if filename not in catalog:
            print(f"  [!] '{filename}' no está en el catálogo. Omitiendo.")
            continue

        rar_path = os.path.join(target_folder, filename)
        if not os.path.exists(rar_path):
            print(f"\nDescargando {filename} desde Zenodo...")
            req = urllib.request.Request(catalog[filename], headers=headers)
            try:
                with urllib.request.urlopen(req) as resp, open(rar_path, "wb") as out_f:
                    total = int(resp.headers.get("Content-Length", 0))
                    downloaded = 0
                    while True:
                        buf = resp.read(1024 * 1024)
                        if not buf:
                            break
                        downloaded += len(buf)
                        out_f.write(buf)
                        mb_d = downloaded / (1024 * 1024)
                        if total > 0:
                            pct = (downloaded / total) * 100
                            sys.stdout.write(f"\r  [⬇] {mb_d:.1f} / {total/(1024*1024):.1f} MB ({pct:.1f}%)")
                        else:
                            sys.stdout.write(f"\r  [⬇] {mb_d:.1f} MB")
                        sys.stdout.flush()
                print(f"\n  [✓] Descargado: {filename}")
            except Exception as e:
                print(f"\n  [X] Error descargando {filename}: {e}")
                if os.path.exists(rar_path):
                    os.remove(rar_path)
                continue
        else:
            print(f"\n[✓] Ya existe: {filename}")

        try:
            metodo = extraer_rar(rar_path, target_folder)
            print(f"  [✓] Descomprimido ({metodo}): {filename}")
        except Exception as e:
            print(f"  [X] Error extrayendo {filename}: {e}")
            if os.path.exists(rar_path):
                os.remove(rar_path)

In [ ]:
# ==============================================================================
# CELDA 4: EXTRACCIÓN DE CARACTERÍSTICAS (71 FEATURES + LABEL)
# ==============================================================================
OP_COLS = ["RPM", "Torque_Nm", "Fuerza_N"]
STAT_COLS = ["RMS", "Kurtosis", "Skewness", "Crest_Factor"]
FFT_COLS = [f"fft_{i}" for i in range(N_FFT_BINS)]
SIGNAL_COLS = STAT_COLS + FFT_COLS
FEATURE_COLS = OP_COLS + SIGNAL_COLS
SCHEMA_COLS = FEATURE_COLS + ["Label"]


def parse_metadata_from_filename(file_path):
    """Extrae condiciones operativas y etiqueta usando solo el nombre del .mat."""
    name = os.path.basename(file_path).upper()
    rpm = 900 if "N09" in name else 1500
    torque = 0.1 if "M01" in name else 0.7
    force = 400 if "F04" in name else 1000

    if "KA" in name:
        fault_type, label = "Pista_Exterior", 1
    elif "KI" in name:
        fault_type, label = "Pista_Interior", 1
    elif "KB" in name:
        fault_type, label = "Combinado", 1
    elif "K0" in name:
        fault_type, label = "Normal", 0
    else:
        fault_type, label = "Anomalia", 1

    return rpm, torque, force, fault_type, label


def extraer_vector_1d(data_obj):
    arr = data_obj
    while isinstance(arr, np.ndarray) and arr.dtype == object and arr.size > 0:
        arr = arr.flat[0]
    return np.ravel(arr).astype(np.float64)


def get_64khz_channels(mat_path, min_len=200000):
    """Extrae canales muestreados a 64 kHz (~256k muestras) del .mat de Paderborn."""
    mat = scipy.io.loadmat(mat_path)
    key = next(k for k in mat.keys() if not k.startswith("__"))
    y_channels = mat[key][0, 0]["Y"]
    n_channels = y_channels.shape[1] if y_channels.ndim > 1 else y_channels.shape[0]

    canales = {"vibration_1": None, "phase_current_1": None, "phase_current_2": None}
    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        ch_name = ""
        if "Name" in ch.dtype.names:
            name_val = ch["Name"]
            while isinstance(name_val, np.ndarray) and name_val.size > 0:
                name_val = name_val.flat[0]
            ch_name = str(name_val).lower()

        signal = extraer_vector_1d(ch["Data"])
        if len(signal) < min_len:
            continue

        if "vib" in ch_name or "acc" in ch_name:
            canales["vibration_1"] = signal
        elif "phase_current_1" in ch_name or "current_1" in ch_name:
            canales["phase_current_1"] = signal
        elif "phase_current_2" in ch_name or "current_2" in ch_name:
            canales["phase_current_2"] = signal

    return canales


def extraer_features_ventana(seg, rpm, torque, force, label):
    """
    Calcula las 71 features con las mismas definiciones que paderborn_features_tratadas.csv:
      - RMS, Kurtosis (Fisher, exceso), Skewness, Crest_Factor (peak / RMS)
      - fft_0..fft_63: |rfft(seg - mean(seg))|[:64] normalizado L2 (antes de log1p)
    """
    rms = float(np.sqrt(np.mean(seg ** 2)))
    std = float(np.std(seg))
    if std > 1e-12:
        kurt = float(kurtosis(seg, fisher=True, bias=True))
        skw = float(skew(seg, bias=True))
    else:
        kurt, skw = 0.0, 0.0
    crest = float(np.max(np.abs(seg)) / (rms + 1e-12))

    seg_centered = seg - np.mean(seg)
    fft_mag = np.abs(np.fft.rfft(seg_centered))[:N_FFT_BINS]
    fft_norm = fft_mag / (np.linalg.norm(fft_mag) + 1e-12)

    rec = {
        "RPM": rpm,
        "Torque_Nm": torque,
        "Fuerza_N": force,
        "RMS": rms,
        "Kurtosis": kurt,
        "Skewness": skw,
        "Crest_Factor": crest,
    }
    for idx, val in enumerate(fft_norm):
        rec[f"fft_{idx}"] = float(val)
    rec["Label"] = int(label)
    return rec


def process_all_64khz_mats(input_dir, canal=CANAL_SENAL, window_size=WINDOW_SIZE, archivos_obj=ARCHIVOS_A_DESCARGAR):
    codigos_obj = {os.path.splitext(f)[0].upper() for f in archivos_obj} if archivos_obj else None
    mat_files = []
    for root, _, files in os.walk(input_dir):
        for f in sorted(files):
            if f.lower().endswith(".mat") and not f.startswith("._"):
                if codigos_obj and not any(c in f.upper() for c in codigos_obj):
                    continue
                mat_files.append(os.path.join(root, f))

    print(f"\n[2/4] Procesando {len(mat_files)} archivos .mat (canal='{canal}')...")
    records, metas = [], []
    omitidos = 0

    for mat_path in mat_files:
        filename = os.path.basename(mat_path)
        rpm, torque, force, fault_type, label = parse_metadata_from_filename(mat_path)

        try:
            canales = get_64khz_channels(mat_path)
            sig = canales.get(canal)
            if sig is None:
                omitidos += 1
                continue

            num_windows = len(sig) // window_size
            for w_idx in range(num_windows):
                seg = sig[w_idx * window_size : (w_idx + 1) * window_size]
                records.append(extraer_features_ventana(seg, rpm, torque, force, label))
                metas.append({
                    "Archivo": filename,
                    "Ventana": w_idx,
                    "Tipo_Falla": fault_type,
                    "RPM": rpm,
                    "Torque_Nm": torque,
                    "Fuerza_N": force,
                    "Label": int(label),
                })
        except Exception as e:
            print(f"  [X] Error procesando {filename}: {e}")

    print(f"  [✓] Ventanas extraídas: {len(records)} | Archivos omitidos (sin canal '{canal}'): {omitidos}")
    df_raw = pd.DataFrame(records, columns=SCHEMA_COLS)
    df_meta = pd.DataFrame(metas)
    return df_raw, df_meta

In [ ]:
# ==============================================================================
# CELDA 5: TRATAMIENTO DE DATOS (LOG1P + CLIPPING + ROBUSTSCALER + PAD + L2)
# ==============================================================================
def zero_pad(X, dim=DIM_CUANTICA):
    pad = dim - X.shape[1]
    if pad < 0:
        raise ValueError(f"Dimensión de features ({X.shape[1]}) > {dim}")
    return np.hstack([X, np.zeros((len(X), pad), dtype=np.float64)]) if pad > 0 else X


def l2_normalize(X):
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(normas == 0, 1.0, normas)


def aplicar_pipeline_tratamiento(df_raw, df_meta, ruta_ref=RUTA_REF, aplicar_clipping=APLICAR_CLIPPING):
    # 1. Cargar referencia sana y validar esquema idéntico
    df_ref = pd.read_csv(ruta_ref)
    assert list(df_ref.columns) == SCHEMA_COLS, (
        f"Esquema distinto al de referencia: {list(df_ref.columns)[:10]} vs {SCHEMA_COLS[:10]}"
    )

    # 2. Limpieza de NaN / Inf manteniendo alineado df_meta
    df = df_raw.replace([np.inf, -np.inf], np.nan).copy()
    mask_valid = ~df.isnull().any(axis=1)
    n_drop = int((~mask_valid).sum())
    if n_drop > 0:
        df = df.loc[mask_valid].reset_index(drop=True)
        df_meta = df_meta.loc[mask_valid].reset_index(drop=True)
        print(f"  [!] Filas eliminadas por NaN/Inf: {n_drop}")

    # 3. Transformación logarítmica log1p sobre las 64 columnas FFT
    df[FFT_COLS] = np.log1p(np.abs(df[FFT_COLS].values))

    # 4. Clipping [P1, P99] usando los límites exactos de la referencia sana
    #    (en df_ref ya fueron recortados al 1%-99%, así que min/max = P1/P99 sanos)
    if aplicar_clipping:
        p_lower = df_ref[SIGNAL_COLS].min(axis=0).values
        p_upper = df_ref[SIGNAL_COLS].max(axis=0).values
        df[SIGNAL_COLS] = np.clip(df[SIGNAL_COLS].values, p_lower, p_upper)
    df_tratado = df[SCHEMA_COLS].copy()

    # 5. RobustScaler ajustado EXCLUSIVAMENTE sobre X_train sano (split 70/15/15, seed=42)
    X_tr_ref, _ = train_test_split(
        df_ref[FEATURE_COLS].values,
        test_size=0.30,
        random_state=RANDOM_STATE,
    )
    scaler = RobustScaler()
    scaler.fit(X_tr_ref)

    X_scaled = scaler.transform(df_tratado[FEATURE_COLS].values)
    df_robust = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
    df_robust["Label"] = df_tratado["Label"].values

    # 6. Zero-padding (71 -> 128) y normalización L2 por fila para Amplitude Encoding (7 qubits)
    #    Nota: como el padding solo agrega ceros, zero_pad -> L2 es equivalente a L2 -> zero_pad.
    X_padded = zero_pad(X_scaled, dim=DIM_CUANTICA)
    X_q = l2_normalize(X_padded)

    return df_tratado, df_robust, X_q, df_meta, df_ref


if __name__ == "__main__":
    print("===========================================================")
    print(" ETL PADERBORN 64 kHz -> FORMATO COMPATIBLE C1 / C3 (7 QUBITS) ")
    print("===========================================================")

    download_and_extract_selected(ARCHIVOS_A_DESCARGAR, ZENODO_CATALOG, TEMP_DIR)
    df_raw, df_meta = process_all_64khz_mats(TEMP_DIR)

    if len(df_raw) == 0:
        raise RuntimeError("No se extrajeron ventanas. Revisa ARCHIVOS_A_DESCARGAR y CANAL_SENAL.")

    print("\n[3/4] Aplicando log1p -> clipping -> RobustScaler -> zero-padding (128) -> L2...")
    df_tratado, df_robust, X_q, df_meta, df_ref = aplicar_pipeline_tratamiento(df_raw, df_meta)

    print("\n[4/4] Guardando artefactos...")
    df_tratado.to_csv(OUT_TRATADAS, index=False)
    df_robust.to_csv(OUT_ROBUST, index=False)
    df_meta.to_csv(OUT_METADATOS, index=False)
    np.savez_compressed(
        OUT_CUANTICO,
        X_q=X_q,
        y=df_tratado["Label"].values,
        tipo_falla=df_meta["Tipo_Falla"].values,
        archivo=df_meta["Archivo"].values,
        feature_cols=np.array(FEATURE_COLS),
    )

    if DRIVE_DIR is not None:
        for ruta_local in (OUT_TRATADAS, OUT_ROBUST, OUT_METADATOS, OUT_CUANTICO):
            shutil.copy2(ruta_local, os.path.join(DRIVE_DIR, os.path.basename(ruta_local)))
        print(f"  [✓] Copiados a Google Drive: {DRIVE_DIR}")

    normas = np.linalg.norm(X_q, axis=1)
    assert list(df_tratado.columns) == list(df_ref.columns)
    assert np.allclose(normas, 1.0)

    print("\n===========================================================")
    print(" RESUMEN FINAL")
    print("===========================================================")
    print(f"  • Tratadas (compatible notebooks): {OUT_TRATADAS} {df_tratado.shape}")
    print(f"  • RobustScaler aplicado:           {OUT_ROBUST} {df_robust.shape}")
    print(f"  • Vectores cuánticos (128, L2=1):  {OUT_CUANTICO} {X_q.shape}")
    print(f"  • Metadatos por ventana:           {OUT_METADATOS} {df_meta.shape}")
    print(f"  • Norma L2 min/max:                {normas.min():.6f} / {normas.max():.6f}")
    print("\nDistribución por Tipo_Falla:")
    print(df_meta["Tipo_Falla"].value_counts().to_string())
    print("\nComparativa rápida de medianas (Referencia sana vs Fallas tratadas):")
    cols_cmp = ["RMS", "Kurtosis", "Skewness", "Crest_Factor", "fft_2", "fft_3"]
    print(pd.DataFrame({"Ref_Sano": df_ref[cols_cmp].median(), "Fallas_ETL": df_tratado[cols_cmp].median()}))

# 📘 Guía de Nomenclatura del Dataset de Paderborn

---

## 1. Prefijo de Estado del Rodamiento (Archivos `.rar`)

| Sigla | Significado en Alemán | Descripción en Español | Tipo de Etiqueta (`Label`) |
| :--- | :--- | :--- | :--- |
| **`K0` / `K00x`** | *Kontrolliert* | **Sano / Normal** (Sin desgaste previo) | `Label 0` |
| **`KA`** | *Kugellager Außenring* | **Falla en Pista Exterior** (*Outer Race Fault*) | `Label 1` |
| **`KI`** | *Kugellager Innenring* | **Falla en Pista Interior** (*Inner Race Fault*) | `Label 1` |
| **`KB`** | *Kombinierter Schaden* | **Falla Combinada** (Múltiples elementos dañados) | `Label 1` |

---

## 2. Clasificación por Método de Daño

### A. Fallas Artificiales (Mecanizadas por EDM / Incisión)
Daños provocados intencionalmente mediante descargas eléctricas o grabado antes del montaje:
* **Pista Exterior (KA):** `KA01.rar`, `KA03.rar`, `KA05.rar`, `KA07.rar`, `KA08.rar`, `KA09.rar`
* **Pista Interior (KI):** `KI01.rar`, `KI03.rar`, `KI05.rar`, `KI07.rar`, `KI08.rar`, `KI09.rar`

### B. Fallas Reales / Naturales (Fatiga / Vida Acelerada)
Daños generados por desprendimiento de material (*pitting*) tras operación continua bajo carga:
* **Pista Exterior (KA):** `KA04.rar`, `KA15.rar`, `KA16.rar`, `KA22.rar`, `KA30.rar`
* **Pista Interior (KI):** `KI04.rar`, `KI14.rar`, `KI16.rar`, `KI18.rar`, `KI21.rar`
* **Combinadas (KB):** `KB23.rar`, `KB24.rar`, `KB27.rar`

---

## 3. Parámetros Operativos (Nomenclatura interna de los `.mat`)

* **`N` (Velocidad de Rotación):** `N09` = 900 RPM (15 Hz) \| `N15` = 1500 RPM (25 Hz)
* **`M` (Torque / Par Motor):** `M01` = 0.1 Nm \| `M07` = 0.7 Nm
* **`F` (Fuerza Radial):** `F04` = 400 N \| `F10` = 1000 N

> **Ejemplo:** `N15_M07_F10_K001_1.mat` $\rightarrow$ Rodamiento sano **K001** a **1500 RPM**, **0.7 Nm** de torque y **1000 N** de fuerza radial.

---

## 4. Artefactos generados por este ETL

- `paderborn_fallas_tratadas.csv`: 72 columnas (`71 features + Label`) tras `log1p` y `clipping [P1, P99]`. Compatible directo con `evaluar_nuevas_fallas()` y con los notebooks `C1` / `C3`.
- `paderborn_fallas_con_robust_scaler.csv`: 72 columnas tras aplicar el `RobustScaler` ajustado sobre `X_train` sano (`random_state=42`).
- `paderborn_fallas_cuantico_128.npz`: arreglo `X_q` de forma `(N, 128)` con zero-padding y normalización $L_2 = 1$, listo para `qml.AmplitudeEmbedding(..., normalize=False)`, junto con `y`, `tipo_falla` y `archivo`.
- `paderborn_fallas_metadatos.csv`: detalle por ventana (`Archivo`, `Ventana`, `Tipo_Falla`, condiciones operativas y `Label`).